# Week 2 – Build a Classification Model
**Vortex Tech AI & ML Internship Track**

**Dataset:** Titanic Passenger Dataset (same dataset cleaned in Week 1)
**Target variable:** `Survived` (0 = did not survive, 1 = survived) — a binary outcome
**Goal:** Train and evaluate a classification model that predicts survival from passenger features.

## 1. Import Libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

## 2. Load and Re-clean the Dataset
Reusing the Week 1 cleaning logic so this notebook can run standalone without depending
on Week 1's output file.

In [2]:
df = pd.read_csv('titanic.csv')

# Week 1 cleaning steps, repeated here so this notebook is self-contained
df['Age'] = df['Age'].fillna(df['Age'].median())
df = df.drop(columns=['Cabin'])
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df = df.drop_duplicates()

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


## 3. Choose Target and Feature Columns

- **Target:** `Survived` (binary: 0/1)
- **Features:** `Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, `Embarked`

Dropped: `PassengerId` (just a row index, no predictive value), `Name` and `Ticket`
(free-text identifiers, not useful without heavier feature engineering).

In [3]:
feature_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target_col = 'Survived'

X = df[feature_cols]
y = df[target_col]

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


## 4. Encode Categorical Features
`Sex` and `Embarked` are text/categorical — models need numbers. We use
`pd.get_dummies()` to one-hot encode them (`drop_first=True` avoids redundant columns).

In [4]:
X = pd.get_dummies(X, columns=['Sex', 'Embarked'], drop_first=True)
X.head()

,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,3,22.0,1,0,7.2500,True,False,True
1,1,38.0,1,0,71.2833,False,False,False
2,3,26.0,0,0,7.9250,False,False,True
3,1,35.0,1,0,53.1000,False,False,True
4,3,35.0,0,0,8.0500,True,False,True


## 5. Train/Test Split (80/20)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows:  {len(X_test)}")

Training rows: 712
Testing rows:  179


## 6. Train a Logistic Regression Model
Logistic Regression is a solid starting point for binary classification — fast to train
and easy to interpret.

In [6]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

log_predictions = log_model.predict(X_test)

## 7. Evaluate Logistic Regression

In [7]:
log_accuracy = accuracy_score(y_test, log_predictions)
log_precision = precision_score(y_test, log_predictions)
log_recall = recall_score(y_test, log_predictions)
log_f1 = f1_score(y_test, log_predictions)

print(f"Accuracy:  {log_accuracy:.3f}")
print(f"Precision: {log_precision:.3f}")
print(f"Recall:    {log_recall:.3f}")
print(f"F1-score:  {log_f1:.3f}")

Accuracy:  0.810
Precision: 0.786
Recall:    0.743
F1-score:  0.764


## 8. Try a Second Model — Decision Tree
The task guide suggests comparing against a Decision Tree if results seem improvable.
We compare both models side by side rather than replacing one with the other.

In [8]:
tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_model.fit(X_train, y_train)

tree_predictions = tree_model.predict(X_test)

tree_accuracy = accuracy_score(y_test, tree_predictions)
tree_precision = precision_score(y_test, tree_predictions)
tree_recall = recall_score(y_test, tree_predictions)
tree_f1 = f1_score(y_test, tree_predictions)

print(f"Accuracy:  {tree_accuracy:.3f}")
print(f"Precision: {tree_precision:.3f}")
print(f"Recall:    {tree_recall:.3f}")
print(f"F1-score:  {tree_f1:.3f}")

Accuracy:  0.799
Precision: 0.828
Recall:    0.649
F1-score:  0.727


## 9. Side-by-Side Comparison

In [9]:
comparison = pd.DataFrame({
    'Logistic Regression': [log_accuracy, log_precision, log_recall, log_f1],
    'Decision Tree': [tree_accuracy, tree_precision, tree_recall, tree_f1]
}, index=['Accuracy', 'Precision', 'Recall', 'F1-score'])

comparison.round(3)

,Logistic Regression,Decision Tree
Accuracy,0.810,0.799
Precision,0.786,0.828
Recall,0.743,0.649
F1-score,0.764,0.727


## 10. Confusion Matrix (Logistic Regression)
A quick look at *where* the model gets it wrong — false positives vs. false negatives.

In [10]:
cm = confusion_matrix(y_test, log_predictions)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: Did Not Survive', 'Actual: Survived'],
    columns=['Predicted: Did Not Survive', 'Predicted: Survived']
)
cm_df

,Predicted: Did Not Survive,Predicted: Survived
Actual: Did Not Survive,90,15
Actual: Survived,19,55


## 11. Summary & Interpretation

**Is the model good?** Both models land in a reasonable range for this dataset — roughly
75-82% accuracy is typical for Titanic with this feature set, which is well above the ~62%
baseline you'd get by always predicting "did not survive" (since that's the majority class).
Precision and recall being reasonably close to accuracy suggests the model isn't wildly
biased toward one class.

**What explains the limitations?** The model only has 7 basic features. It has no
information from `Name` (titles like "Mrs." or "Master" correlate with age/social status),
`Ticket`, or `Cabin` deck — all of which carried real signal about survival but were
dropped or removed during cleaning. `Fare` and `Pclass` also overlap heavily (they both
capture socio-economic status), so the model has some redundant rather than independent
information.

**What would we try next?**
1. **Feature engineering** — extract passenger title from `Name`, and a `FamilySize`
   feature from `SibSp` + `Parch`.
2. **More data / cross-validation** — a single 80/20 split can be noisy; k-fold
   cross-validation would give a more stable estimate of true performance.
3. **A stronger model** — Random Forest or Gradient Boosting typically outperform a
   single Decision Tree on tabular data like this, at the cost of interpretability.
